# 01 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [ ]:
from pathlib import Path

import pandas as pd

from amr.filtering import (
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines.yearly_collection import (
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [ ]:
PATHOGENS_PATH = Path("bacteria_synonym_lookup.csv")
ANTIBIOTIC_METADATA_PATH = Path("antibiotic_list2.xlsx")

OUTPUT_YEARLY_AMR_PATH = Path("yearly_amr_pairs.parquet")
OUTPUT_YEARLY_SCREEN_PATH = Path("yearly_amr_screened_pairs.parquet")
OUTPUT_UNIFIED_DECISIONS_PATH = Path("yearly_amr_unified_decisions.parquet")
OUTPUT_PATHOGEN_BG_PATH = Path("pathogen_tiab_counts.parquet")
OUTPUT_ANTIBIOTIC_BG_PATH = Path("antibiotic_tiab_counts.parquet")

INTERVAL_PMIDS_PATH = Path("pmids_results.parquet")
YEARLY_PAIR_PMID_PATH = Path("yearly_pair_pmid.parquet")
PUBMED_METADATA_CACHE_PATH = Path("pubmed_pmid_metadata.parquet")
ABSTRACT_CACHE_PATH = Path("pmid_abstracts.parquet")

PUBMED_EMAIL = "your_email@example.com"  # replace with a real contact email

# Partition-friendly execution window: run a slice now, merge into canonical stores.
SLICE_START_YEAR = 2020
SLICE_END_YEAR = 2026
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"

INTERVAL_YEAR_GAP = 5

# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    '"antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]'
]

RUN_LLM_FILTER = False  # set True to execute LLM screening for this slice
MAX_LLM_CALLS = None


## Load Pathogen and Antibiotic Lists


In [ ]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_excel(ANTIBIOTIC_METADATA_PATH)

pathogens = sorted(bacteria_synonym_df["bacteria name"].dropna().astype(str).str.strip().unique())
antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

print(f"Pathogens: {len(pathogens):,}")
print(f"Antibiotics: {len(antibiotics):,}")


## Collect Yearly Pair Dataset and Background Marginals


In [ ]:
yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
)

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")



## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


In [ ]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

resume_df = None
if OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)

if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model="gpt-4o-mini", max_tokens=256, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=True,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        verbose=1,
    )
else:
    unified_df = resume_df if resume_df is not None else pd.DataFrame()
    if unified_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Run with RUN_LLM_FILTER=True at least once."
        )

print(unified_df["unified_decision"].value_counts(dropna=False))


## Export Canonical Screened Yearly Dataset
Includes decisions `yes` + `uncertain` by default.


In [ ]:
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=("yes", "uncertain"),
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=True,
)

print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)

